<a href="https://colab.research.google.com/github/shriharish123/DAA-lab-exercise/blob/main/DAA_Lab_exercise5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import time
import random
import string


def naive_search(text, pattern):
    n, m = len(text), len(pattern)
    matches, comparisons = [], 0

    for i in range(n - m + 1):
        j = 0
        while j < m:
            comparisons += 1
            if text[i + j] != pattern[j]:
                break
            j += 1

        if j == m:
            matches.append(i)

    return matches, comparisons


def compute_lps(pattern):
    m = len(pattern)
    lps = [0] * m
    length, i = 0, 1

    while i < m:
        if pattern[i] == pattern[length]:
            length += 1
            lps[i] = length
            i += 1
        elif length != 0:
            length = lps[length - 1]
        else:
            lps[i] = 0
            i += 1

    return lps


def kmp_search(text, pattern):
    n, m = len(text), len(pattern)
    lps = compute_lps(pattern)
    matches, comparisons = [], 0

    i = j = 0

    while i < n:
        comparisons += 1

        if pattern[j] == text[i]:
            i += 1
            j += 1

            if j == m:
                matches.append(i - j)
                j = lps[j - 1]

        elif i < n and pattern[j] != text[i]:
            if j != 0:
                j = lps[j - 1]
            else:
                i += 1

    return matches, comparisons


def rabin_karp(text, pattern, q=101):
    n, m = len(text), len(pattern)
    d = 256
    h = pow(d, m - 1, q)

    p_hash = 0
    t_hash = 0
    matches, comparisons = [], 0

    for i in range(m):
        p_hash = (d * p_hash + ord(pattern[i])) % q
        t_hash = (d * t_hash + ord(text[i])) % q

    for s in range(n - m + 1):
        if p_hash == t_hash:
            for k in range(m):
                comparisons += 1
                if text[s + k] != pattern[k]:
                    break
            else:
                matches.append(s)

        if s < n - m:
            t_hash = (d * (t_hash - ord(text[s]) * h) + ord(text[s + m])) % q

            if t_hash < 0:
                t_hash += q

    return matches, comparisons


# ---------------- Main Execution ----------------

text = "AABAACAADAABAABA"
pattern = "AABA"

print(f"Text: {text}")
print(f"Pattern: {pattern}")

m1, c1 = naive_search(text, pattern)
m2, c2 = kmp_search(text, pattern)
m3, c3 = rabin_karp(text, pattern)

print(f"\nNaive -> Matches at: {m1}, Comparisons: {c1}")
print(f"KMP   -> Matches at: {m2}, Comparisons: {c2}")
print(f"RK    -> Matches at: {m3}, Comparisons: {c3}")


# ---------------- Performance Comparison ----------------

text_large = ''.join(random.choices('ABCD', k=10000))
patterns = ['AB', 'ABCD', 'ABCDAB', 'ABCDABCD']

print(f"\n{'Pattern':>12} {'Naive':>10} {'KMP':>10} {'RK':>10}")
print("-" * 50)

for p in patterns:
    _, c1 = naive_search(text_large, p)
    _, c2 = kmp_search(text_large, p)
    _, c3 = rabin_karp(text_large, p)

    print(f"{p:>12} {c1:>10} {c2:>10} {c3:>10}")

Text: AABAACAADAABAABA
Pattern: AABA

Naive -> Matches at: [0, 9, 12], Comparisons: 30
KMP   -> Matches at: [0, 9, 12], Comparisons: 20
RK    -> Matches at: [0, 9, 12], Comparisons: 12

     Pattern      Naive        KMP         RK
--------------------------------------------------
          AB      12504      11888       1234
        ABCD      13299      12456        268
      ABCDAB      13356      12504        121
    ABCDABCD      13355      12505        150


In [8]:
import heapq

import heapq

class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]

    def union(self, x, y):
        rx = self.find(x)
        ry = self.find(y)

        if rx == ry:
            return False

        if self.rank[rx] < self.rank[ry]:
            rx, ry = ry, rx

        self.parent[ry] = rx

        if self.rank[rx] == self.rank[ry]:
            self.rank[rx] += 1

        return True

def kruskal(n, edges):
    """edges: list of (weight, u, v)"""

    edges.sort()

    uf = UnionFind(n)
    mst = []
    cost = 0

    for w, u, v in edges:
        if uf.union(u, v):
            mst.append((u, v, w))
            cost += w

            if len(mst) == n - 1:
                break

    return mst, cost

def prim(n, adj, start=0):
    """adj: adjacency list {u: [(v, w), ...]}"""

    INF = float('inf')
    key = [INF] * n
    parent = [-1] * n
    inMST = [False] * n

    key[start] = 0
    pq = [(0, start)]

    mst = []
    cost = 0

    while pq:
        w, u = heapq.heappop(pq)

        if inMST[u]:
            continue

        inMST[u] = True

        if parent[u] != -1:
            mst.append((parent[u], u, w))
            cost += w

        for v, wt in adj.get(u, []):
            if not inMST[v] and wt < key[v]:
                key[v] = wt
                parent[v] = u
                heapq.heappush(pq, (wt, v))

    return mst, cost

n = 7

edges = [
    (7, 0, 1),
    (5, 0, 3),
    (8, 1, 2),
    (9, 1, 3),
    (7, 1, 4),
    (5, 2, 4),
    (15, 3, 4),
    (6, 3, 5),
    (8, 4, 5),
    (9, 4, 6),
    (11, 5, 6)
]
adj = {}

for w, u, v in edges:
    adj.setdefault(u, []).append((v, w))
    adj.setdefault(v, []).append((u, w))

k_mst, k_cost = kruskal(n, edges[:])
p_mst, p_cost = prim(n, adj)

print("=== Kruskal's MST ===")
for u, v, w in k_mst:
    print(f"Edge ({u} - {v}) Weight: {w}")

print(f"Total MST Cost: {k_cost}")

print("\n=== Prim's MST ===")
for u, v, w in p_mst:
    print(f"Edge ({u} - {v}) Weight: {w}")

print(f"Total MST Cost: {p_cost}")

=== Kruskal's MST ===
Edge (0 - 3) Weight: 5
Edge (2 - 4) Weight: 5
Edge (3 - 5) Weight: 6
Edge (0 - 1) Weight: 7
Edge (1 - 4) Weight: 7
Edge (4 - 6) Weight: 9
Total MST Cost: 39

=== Prim's MST ===
Edge (0 - 3) Weight: 5
Edge (3 - 5) Weight: 6
Edge (0 - 1) Weight: 7
Edge (1 - 4) Weight: 7
Edge (4 - 2) Weight: 5
Edge (4 - 6) Weight: 9
Total MST Cost: 39


In [9]:
import heapq


def dijkstra(graph, source):
    """
    Dijkstra's Algorithm using Min-Heap
    Time: O((V + E) log V)
    Space: O(V)

    graph: dict {u: [(v, weight), ...]}
    """

    n = len(graph)
    dist = [float('inf')] * n
    prev = [None] * n

    dist[source] = 0
    pq = [(0, source)]  # (distance, vertex)
    visited = set()


    while pq:
        d, u = heapq.heappop(pq)

        if u in visited:
            continue

        visited.add(u)

        for v, w in graph[u]:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                prev[v] = u
                heapq.heappush(pq, (dist[v], v))

    return dist, prev


def reconstruct_path(prev, source, target):
    path = []
    node = target

    while node is not None:
        path.append(node)
        node = prev[node]

    path.reverse()

    if path and path[0] == source:
        return path

    return []

graph = {
    0: [(1, 4), (2, 1)],
    1: [(3, 1)],
    2: [(1, 2), (3, 5)],
    3: [(4, 3)],
    4: [(5, 2)],
    5: []
}

source = 0

dist, prev = dijkstra(graph, source)

print(f"Shortest paths from vertex {source}:\n")
print(f"{'Vertex':>8} {'Distance':>10} {'Path':>30}")
print("-" * 55)

for v in range(len(graph)):
    path = reconstruct_path(prev, source, v)
    path_str = " -> ".join(map(str, path)) if path else "No path"
    d = dist[v] if dist[v] != float('inf') else "INF"

    print(f"{v:>8} {str(d):>10} {path_str:>30}")

Shortest paths from vertex 0:

  Vertex   Distance                           Path
-------------------------------------------------------
       0          0                              0
       1          3                    0 -> 2 -> 1
       2          1                         0 -> 2
       3          4               0 -> 2 -> 1 -> 3
       4          7          0 -> 2 -> 1 -> 3 -> 4
       5          9     0 -> 2 -> 1 -> 3 -> 4 -> 5


In [10]:
import random

comparison_count = 0  # Global counter


def min_max_dc(arr, low, high):
    global comparison_count

    # Base case: single element
    if low == high:
        return arr[low], arr[low]

    # Base case: two elements
    if high == low + 1:
        comparison_count += 1
        if arr[low] < arr[high]:
            return arr[low], arr[high]
        else:
            return arr[high], arr[low]

    # Divide
    mid = (low + high) // 2

    lmin, lmax = min_max_dc(arr, low, mid)
    rmin, rmax = min_max_dc(arr, mid + 1, high)

    # Combine
    comparison_count += 1
    overall_min = lmin if lmin < rmin else rmin

    comparison_count += 1
    overall_max = lmax if lmax > rmax else rmax

    return overall_min, overall_max


def min_max_naive(arr):
    mn = mx = arr[0]
    comps = 0

    for x in arr[1:]:
        comps += 1
        if x < mn:
            mn = x

        comps += 1
        if x > mx:
            mx = x

    return mn, mx, comps


# ---------- Demonstration ----------

arr = [3, 1, 7, 4, 9, 2, 8, 5, 6, 0]

comparison_count = 0
mn, mx = min_max_dc(arr, 0, len(arr) - 1)
dc_comps = comparison_count

_, _, naive_comps = min_max_naive(arr)

print(f"Array: {arr}")
print(f"Min: {mn}, Max: {mx}")
print(f"D&C Comparisons: {dc_comps}")
print(f"Naive Comparisons: {naive_comps}")


# ---------- Performance Analysis ----------

print(f"\n{'Size':>8} {'DC Comps':>12} {'Naive Comps':>14} {'Formula 3n/2-2':>16}")
print("-" * 56)

for size in [10, 100, 1000, 10000]:
    arr = [random.randint(1, 10000) for _ in range(size)]

    comparison_count = 0
    mn, mx = min_max_dc(arr, 0, len(arr) - 1)
    dc = comparison_count

    _, _, naive = min_max_naive(arr)

    formula = (3 * size) // 2 - 2

    print(f"{size:>8} {dc:>12} {naive:>14} {formula:>16}")

Array: [3, 1, 7, 4, 9, 2, 8, 5, 6, 0]
Min: 0, Max: 9
D&C Comparisons: 14
Naive Comparisons: 18

    Size     DC Comps    Naive Comps   Formula 3n/2-2
--------------------------------------------------------
      10           14             18               13
     100          162            198              148
    1000         1510           1998             1498
   10000        15902          19998            14998
